# 面试题：搜索拼写纠错与 Query Rewrite 应该怎样设计？

“做编辑距离”只是第一步。一个可上线方案要回答：词表从哪里来、候选如何低成本生成、键盘/转置错误怎样建模、上下文如何消歧、什么时候拒绝改写、数字和实体怎样保护、如何评估误改率，以及词表升级如何版本化。

本 Notebook 只用 Python/NumPy 手写 Trie、动态规划距离、noisy-channel 打分、bigram 上下文、置信 margin、短语改写和发布快照，不调用拼写纠错库。

In [ ]:
import copy,hashlib,json,math,re,unicodedata,warnings
from collections import Counter,defaultdict
from dataclasses import dataclass
from types import MappingProxyType
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")
import numpy as np
def canonical70(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(",",":"))
def sha70(x): return hashlib.sha256(x).hexdigest()
assert np.isfinite([0.]).all()

## 1. 词表、频率与保护规则

词表不是字典文件，而是经过清洗的查询/文档频率快照。这里给每个词 unigram count，并从已审核 session 构造 bigram count。纯数字、版本号、SKU 和显式保护实体默认不改，避免把 `gpt-4`、订单号或品牌改成高频普通词。

频率窗口、租户和语言必须绑定版本；攻击流量若直接进入词表，可能把恶意拼写“教”成正确词。

In [ ]:
VOCAB70={"search":900,"engine":700,"retrieval":650,"vector":600,"keyword":560,"ranking":500,"model":480,"query":450,"rewrite":300,"document":420,"classification":260,"graph":350,"knowledge":330,"token":400}
SESSIONS70=["search engine","vector search","keyword search","query rewrite","retrieval model","document retrieval","knowledge graph","ranking model"]
PROTECTED70={"gpt-4","bm25","hnsw"}
def tokenize70(text): return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)?",unicodedata.normalize("NFKC",text).lower())
bigram70=Counter()
for row in SESSIONS70:
    toks=["<s>"]+tokenize70(row)
    bigram70.update(zip(toks,toks[1:]))
assert tokenize70("Vector SEARCH, GPT-4")==["vector","search","gpt-4"]
assert bigram70[("vector","search")]==1 and VOCAB70["search"]>VOCAB70["rewrite"]
assert PROTECTED70.isdisjoint(VOCAB70)

## 2. 手写 Damerau–Levenshtein 距离

普通 Levenshtein 支持插入、删除、替换；搜索常见相邻键转置，如 `serach`，因此加入相邻 transposition。动态规划状态 `dp[i,j]` 表示前缀最小代价，复杂度 `O(|a||b|)`。

可以给键盘邻键、同音字或中英文混输更低代价，但成本矩阵必须从错误日志估计，不能凭感觉调到只对 demo 有效。

In [ ]:
def damerau70(a,b):
    if not isinstance(a,str) or not isinstance(b,str): raise TypeError("distance_string_contract")
    dp=np.zeros((len(a)+1,len(b)+1),dtype=np.int32); dp[:,0]=np.arange(len(a)+1); dp[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            cost=0 if a[i-1]==b[j-1] else 1
            dp[i,j]=min(dp[i-1,j]+1,dp[i,j-1]+1,dp[i-1,j-1]+cost)
            if i>1 and j>1 and a[i-1]==b[j-2] and a[i-2]==b[j-1]: dp[i,j]=min(dp[i,j],dp[i-2,j-2]+1)
    return int(dp[-1,-1])
assert damerau70("search","search")==0 and damerau70("serach","search")==1
assert damerau70("retrival","retrieval")==1 and damerau70("vector","vectro")==1
assert damerau70("","abc")==3 and damerau70("abc","")==3
assert damerau70("abc","xyz")==3 and damerau70("a","ab")==1
try: damerau70(None,"a"); raise AssertionError("non-string distance accepted")
except TypeError as e: assert str(e)=="distance_string_contract"

## 3. Trie 上的候选生成

对词表逐词计算距离是 `O(|V|L²)`。Trie 共享前缀；沿边更新一行 Levenshtein DP，若该行最小值已超过阈值就剪枝。这里候选阶段用标准 Levenshtein，最终再用带转置距离精排，因此 `serach` 需要允许候选阈值 2。

生产还可使用 SymSpell delete index、WFST 或分语言索引。候选生成要有限额，防止超长垃圾 query 放大 CPU。

In [ ]:
class TrieNode70:
    def __init__(self): self.children={}; self.word=None
class Trie70:
    def __init__(self,words):
        self.root=TrieNode70()
        for word in sorted(words):
            node=self.root
            for ch in word: node=node.children.setdefault(ch,TrieNode70())
            node.word=word
    def within(self,token,max_dist=2,limit=32):
        if not token or max_dist<0 or limit<1: raise ValueError("candidate_contract")
        initial=list(range(len(token)+1)); out=[]
        def visit(node,ch,prev):
            row=[prev[0]+1]
            for j,tch in enumerate(token,1): row.append(min(row[-1]+1,prev[j]+1,prev[j-1]+(ch!=tch)))
            if node.word is not None and row[-1]<=max_dist: out.append((node.word,row[-1]))
            if min(row)<=max_dist and len(out)<limit:
                for nxt,child in sorted(node.children.items()): visit(child,nxt,row)
        for ch,node in sorted(self.root.children.items()): visit(node,ch,initial)
        return out[:limit]
trie70=Trie70(VOCAB70)
cand70=dict(trie70.within("retrival",2))
assert "retrieval" in cand70 and cand70["retrieval"]==1
assert "search" in dict(trie70.within("serach",2))
assert trie70.within("zzzzzz",1)==[]
try: trie70.within("",2); raise AssertionError("empty candidate token accepted")
except ValueError as e: assert str(e)=="candidate_contract"

## 4. Noisy-channel 与上下文打分

目标近似为 `log P(candidate) + log P(candidate|previous) - λ*edit_cost`。unigram 提供先验，bigram 给上下文，编辑距离刻画错误通道。加一平滑防止 unseen bigram 为负无穷。

高频词不能无条件碾压：若原词已在词表，默认保留；候选第一名与第二名 margin 不足时拒绝自动改写，只给 suggestion。

In [ ]:
TOTAL70=sum(VOCAB70.values()); BIGRAM_V70=len(VOCAB70)+1
def score_candidate70(token,candidate,previous="<s>"):
    edit=damerau70(token,candidate); uni=math.log((VOCAB70[candidate]+1)/(TOTAL70+len(VOCAB70)))
    context=math.log((bigram70[(previous,candidate)]+1)/(sum(v for (p,_),v in bigram70.items() if p==previous)+BIGRAM_V70))
    return uni+.8*context-1.6*edit
assert score_candidate70("serach","search")>score_candidate70("serach","graph")
assert score_candidate70("serach","search","vector")>score_candidate70("serach","search","knowledge")
ranked70=sorted(((w,score_candidate70("vectro",w)) for w,_ in trie70.within("vectro",2)),key=lambda z:(-z[1],z[0]))
assert ranked70[0][0]=="vector" and np.isfinite([s for _,s in ranked70]).all()

## 5. Token 决策、拒绝改写与完整 Query Rewrite

每个 token 返回原词、候选、距离、margin 和 action。词表内、保护词、纯数字直接 KEEP；无候选 ABSTAIN；高 margin 才 AUTO_CORRECT；低 margin 只 SUGGEST。完整 query 逐 token 处理，并让上一个最终词进入 bigram 上下文。

线上 UI 必须区分“已自动改写”和“你是否想搜”，同时保留原 query 供日志归因和一键回退。

In [ ]:
@dataclass(frozen=True)
class Decision70: original:str; output:str; action:str; distance:int; margin:float
def correct_token70(token,previous="<s>",auto_margin=.7):
    if token in VOCAB70 or token in PROTECTED70 or token.isdigit(): return Decision70(token,token,"KEEP",0,float("inf"))
    candidates={w for w,_ in trie70.within(token,2)}
    if not candidates: return Decision70(token,token,"ABSTAIN",99,0.)
    rows=sorted([(score_candidate70(token,w,previous),w) for w in candidates],key=lambda z:(-z[0],z[1])); best_score,best=rows[0]; second=rows[1][0] if len(rows)>1 else best_score-10
    margin=best_score-second; action="AUTO_CORRECT" if margin>=auto_margin else "SUGGEST"
    return Decision70(token,best if action=="AUTO_CORRECT" else token,action,damerau70(token,best),margin)
def rewrite70(text):
    tokens=tokenize70(text)
    if not tokens or len(tokens)>12: raise ValueError("query_length_contract")
    decisions=[]; previous="<s>"
    for token in tokens:
        d=correct_token70(token,previous); decisions.append(d); previous=d.output
    return " ".join(d.output for d in decisions),tuple(decisions)
rewrite_probe70,decisions_probe70=rewrite70("vectro serach")
assert rewrite_probe70=="vector search" and all(d.action=="AUTO_CORRECT" for d in decisions_probe70)
assert rewrite70("gpt-4 search 2026")[0]=="gpt-4 search 2026"
assert correct_token70("zzzz").action=="ABSTAIN" and correct_token70("search").action=="KEEP"
try: rewrite70(" "); raise AssertionError("empty rewrite accepted")
except ValueError as e: assert str(e)=="query_length_contract"

## 6. 离线评估：修对、误改与覆盖率

gold 至少区分 typo 应改、正确 query 不应改、实体保护和无解拒绝。只看 correction accuracy 会鼓励系统把所有 query 都改掉；需要同时报告 auto coverage、auto precision、false-correction rate 和 suggestion 接受率。

query family 与用户模板要 group split，不能把同一错拼的轻微变体分到训练和测试两侧。

In [ ]:
eval70=[("serach engine","search engine"),("retrival model","retrieval model"),("vector search","vector search"),("gpt-4 serach","gpt-4 search"),("zzzz query","zzzz query")]
rows70=[(raw,rewrite70(raw)[0],gold) for raw,gold in eval70]
accuracy70=np.mean([pred==gold for _,pred,gold in rows70]); changed70=[(raw,pred,gold) for raw,pred,gold in rows70 if pred!=raw]
precision70=np.mean([pred==gold for _,pred,gold in changed70])
assert accuracy70==1. and precision70==1. and len(changed70)==3
assert any(raw==pred for raw,pred,_ in rows70) and any(raw!=pred for raw,pred,_ in rows70)
assert len({raw for raw,_,_ in rows70})==len(rows70)

## 7. 热词更新、回滚与缓存键

新词进入词表不能原地修改旧 Trie：构建新版本，跑保护词与误改回归，再原子切换。缓存键必须包含 lexicon/context/model 版本，否则升级后会命中旧改写。紧急回滚只切版本指针，不重放用户请求。

个性化上下文要受权限与隐私约束；本例使用全局审核 bigram，不处理用户画像。

In [ ]:
vocab_v2_70=copy.deepcopy(VOCAB70); vocab_v2_70["agentic"]=200
trie_v2_70=Trie70(vocab_v2_70)
assert "agentic" in dict(trie_v2_70.within("agentci",2)) and "agentic" not in dict(trie70.within("agentci",2))
cache_key_v1_70=sha70(canonical70({"query":"agentci","lexicon":"v1","context":"b1"}).encode())
cache_key_v2_70=sha70(canonical70({"query":"agentci","lexicon":"v2","context":"b1"}).encode())
assert cache_key_v1_70!=cache_key_v2_70 and len(cache_key_v1_70)==64

## 8. 发布清单与面试回答结构

manifest 绑定完整词表频率、bigram、保护词、Unicode/analyzer、候选阈值、打分权重和测试摘要。loader 从实际结构重算整体 digest，与包外 registry 比对；不能只信包内 `vocab_hash`。

面试回答顺序：错误类型与 SLO → 候选生成 → noisy-channel/context → 拒绝与保护 → 评估 → 热更新/监控。要明确自动改写的风险通常比漏改更高。

In [ ]:
manifest70={"artifact_id":"spell-rewrite-v1","vocab":VOCAB70,"bigram":[[a,b,c] for (a,b),c in sorted(bigram70.items())],"protected":sorted(PROTECTED70),"analyzer":"NFKC-lower-regex-v1","candidate":{"trie_distance":2,"limit":32},"score":{"context_weight":.8,"edit_weight":1.6,"auto_margin":.7},"eval_digest":sha70(canonical70(rows70).encode())}
TRUST70=MappingProxyType({manifest70["artifact_id"]:sha70(canonical70(manifest70).encode())})
def load_rewriter70(m):
    actual=copy.deepcopy(m)
    if TRUST70.get(actual.get("artifact_id"))!=sha70(canonical70(actual).encode()): raise RuntimeError("untrusted_rewriter")
    if actual["vocab"]!=VOCAB70 or actual["protected"]!=sorted(PROTECTED70): raise RuntimeError("rewriter_protocol")
    return MappingProxyType(actual)
published70=load_rewriter70(manifest70)
assert published70["score"]["auto_margin"]==.7 and isinstance(TRUST70,MappingProxyType)
forged70=copy.deepcopy(manifest70); forged70["score"]["auto_margin"]=-1
try: load_rewriter70(forged70); raise AssertionError("forged rewriter accepted")
except RuntimeError as e: assert str(e)=="untrusted_rewriter"
print({"accuracy":accuracy70,"auto_precision":precision70,"examples":rows70[:2]})

## 9. 失败模式、复杂度与来源

Trie DP 最坏仍与访问节点数乘 token 长度相关，因此必须限制 query、距离、候选和超时。常见故障：高频词过改、实体/数字损坏、词表投毒、跨语言误改、上下文泄漏、升级缓存未失效和只测 typo 不测正确 query。

- Brill & Moore, [An Improved Error Model for Noisy Channel Spelling Correction](https://aclanthology.org/P00-1037/), ACL 2000。
- Norvig, [How to Write a Spelling Corrector](https://norvig.com/spell-correct.html)，简洁 noisy-channel 背景。
- Jurafsky & Martin, [Speech and Language Processing: Spelling Correction](https://web.stanford.edu/~jurafsky/slp3/)，语言模型与错误通道背景。